# Sentiment Analysis of Amazon Product Reviews using RoBERTa Transformer Model 

## 1. Setup and Imports

In [ ]:
! pip install ipykernel ipywidgets pyarrow pandas numpy nltk matplotlib seaborn scikit-learn torch torchvision torchaudio transformers

In [ ]:
! pip list

In [ ]:
import os
import numpy as np
import pandas as pd
import re
import warnings

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

nltk.download()

warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Load Dataset

In [ ]:
train_path = "./amazon_review_polarity_csv/train.csv"
test_path = "./amazon_review_polarity_csv/test.csv"

df_original_train = pd.read_csv(
    train_path,
    header=None,
    names=("polarity", "title", "text"),
    nrows=200000 # Reading the first 200K rows out of 3.6M
)
df_original_test  = pd.read_csv(
    test_path,
    header=None,
    names=("polarity", "title", "text"),
    nrows=200000 # Reading the first 200K rows out of 400K
)

# Map new polarity 1 becomes 0 (negative review), 2 becomes 1 (positive review)
df_original_train["polarity"] = df_original_train["polarity"].map({1: 0, 2: 1})
df_original_test["polarity"]  = df_original_test["polarity"].map({1: 0, 2: 1})

# Drop rows where title or text is missing
df_original_train = df_original_train.dropna(subset=["title", "text"])
df_original_test  = df_original_test.dropna(subset=["title", "text"])

# Keeping only rows where title and text are strings (extra safety)
mask_train = df_original_train["title"].apply(lambda x: isinstance(x, str)) & \
             df_original_train["text"].apply(lambda x: isinstance(x, str))
mask_test  = df_original_test["title"].apply(lambda x: isinstance(x, str)) & \
             df_original_test["text"].apply(lambda x: isinstance(x, str))

df_original_train = df_original_train[mask_train]
df_original_test  = df_original_test[mask_test]

# Random Samples for model training
# Random Sample of 9K out of 200K
df_train_sample = df_original_train.sample(9000, random_state=42)
# Random Sample of 1K out of 200K
df_test_sample  = df_original_test.sample(1000, random_state=42) 

df_train_sample.head()

In [ ]:
df_test_sample.head()

## 3. Baseline: Logistic Regression with TF-IDF

### 3.1 Text Preprocessing for TF-IDF

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    tokens = nltk.word_tokenize(text)
    # Remove punctuation (keep only alphanumeric tokens)
    tokens = [token for token in tokens if re.match(r"^\w+$", token)]
    # Remove stopwords
    stop_words = set(stopwords.words("english"))
    tokens = [token for token in tokens if token not in stop_words]
    # Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]
    return " ".join(tokens)

def preprocess_for_logistic(df):
    df = df.copy()
    df["clean_text"] = df["title"] + " " + df["text"]
    df["clean_text"] = df["clean_text"].apply(preprocess_text)
    return df

df_train_logistic = preprocess_for_logistic(df_train_sample)
df_test_logistic  = preprocess_for_logistic(df_test_sample)

df_train_logistic.head()

In [ ]:
#Create Training and Test sets for Logistic Regression model
X_train_logistic = df_train_logistic["clean_text"]
y_train_logistic = df_train_logistic["polarity"]

X_test_logistic = df_test_logistic["clean_text"]
y_test_logistic = df_test_logistic["polarity"]

### 3.2 TF-IDF Vectorization

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_train_vec = tfidf_vectorizer.fit_transform(X_train_logistic)
X_test_vec  = tfidf_vectorizer.transform(X_test_logistic)

### 3.3 Logistic Regression Training and Evaluation

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vec, y_train_logistic)

y_pred_lr = lr.predict(X_test_vec)

accuracy_lr = accuracy_score(y_test_logistic, y_pred_lr)
print("Logistic Regression Accuracy:", accuracy_lr)
print("\nClassification Report (Logistic Regression):")
print(classification_report(y_test_logistic, y_pred_lr, target_names=["Negative", "Positive"]))

cm_lr = confusion_matrix(y_test_logistic, y_pred_lr)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()

## 4. RoBERTa Fine-Tuning

### 4.1 Prepare Data for RoBERTa

In [ ]:
df_train_roberta = df_train_sample.copy()
df_test_roberta  = df_test_sample.copy()

In [ ]:
# Concatenate title and text into a single field
df_train_roberta["joined_text"] = df_train_roberta["title"] + " " + df_train_roberta["text"]
df_test_roberta["joined_text"]  = df_test_roberta["title"] + " " + df_test_roberta["text"]

X_train_roberta = df_train_roberta["joined_text"].tolist()
y_train_roberta = df_train_roberta["polarity"].tolist()

X_test_roberta  = df_test_roberta["joined_text"].tolist()
y_test_roberta  = df_test_roberta["polarity"].tolist()

len(X_train_roberta), len(X_test_roberta)

### 4.2 Tokenizer and Dataset Class

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

class ReviewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}  # remove batch dim
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

train_dataset = ReviewsDataset(X_train_roberta, y_train_roberta, tokenizer)
test_dataset  = ReviewsDataset(X_test_roberta,  y_test_roberta,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

### 4.3 Initialize RoBERTa Model

In [ ]:
from transformers import RobertaConfig, RobertaForSequenceClassification

config = RobertaConfig.from_pretrained(
    "roberta-base",
    num_labels=2, # binary classification
    output_attentions=False,    
    output_hidden_states=False,
    attn_implementation="eager"
)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    config=config
).to(device)


### 4.4 Training Loop

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

num_epochs = 5 #Small number of epochs due to large train time per epoch

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Training loss: {avg_loss:.4f}")

### 4.5 Evaluation on Test Set

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds  = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

accuracy_roberta = accuracy_score(all_labels, all_preds)
print("RoBERTa Accuracy:", accuracy_roberta)
print("\nClassification Report (RoBERTa):")
print(classification_report(all_labels, all_preds, target_names=["Negative", "Positive"]))

cm_roberta = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_roberta, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("RoBERTa - Confusion Matrix")
plt.show()

### 4.6 Attention Visualization

In [ ]:
def visualize_attention(text, model, tokenizer, device, layer_index=-1, max_len=32):
    """
    Visualize average attention for a single input text at a given layer,
    focusing only on inner tokens (ignoring <s> and </s>).
    """

    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_len
    )
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
            return_dict=True
        )

    # attentions: tuple(num_layers) of (batch, num_heads, seq_len, seq_len)
    attentions = outputs.attentions
    attn_layer = attentions[layer_index][0]  # (num_heads, seq_len, seq_len)

    # Average across heads
    attn_mean = attn_layer.mean(dim=0).cpu().numpy()  # (seq_len, seq_len)

    # Tokens and valid length
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    valid_len = int(attention_mask[0].sum().item())
    tokens = tokens[:valid_len]
    attn_mean = attn_mean[:valid_len, :valid_len]

    # Focus on inner tokens only (skip <s> at index 0 and </s> at last index)
    if valid_len > 2:
        inner_idx = list(range(1, valid_len - 1))
        inner_tokens = [tokens[i] for i in inner_idx]
        attn_inner = attn_mean[np.ix_(inner_idx, inner_idx)]
    else:
        inner_tokens = tokens
        attn_inner = attn_mean

    plt.figure(figsize=(8, 6))
    plt.imshow(attn_inner, cmap="viridis")
    plt.colorbar()
    plt.xticks(range(len(inner_tokens)), inner_tokens, rotation=90)
    plt.yticks(range(len(inner_tokens)), inner_tokens)
    plt.title(f"RoBERTa Attention Heatmap (Layer {layer_index}, inner tokens)")
    plt.tight_layout()
    plt.show()


In [ ]:
sample_text = df_test_roberta["joined_text"].iloc[9]
print("TEXT:\n", sample_text)
visualize_attention(sample_text, model, tokenizer, device, layer_index=-1, max_len=30)

### 4.7 Visualize Specific Token Attention

In [ ]:
def visualize_token_attention(text, model, tokenizer, device,
                              target_word="product", layer_index=-1, max_len=32):
    """
    Show how a specific token (e.g. 'product') attends to all other tokens
    at a given layer, averaged over heads.
    """

    model.eval()
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_len
    )
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
            return_dict=True
        )

    attentions = outputs.attentions
    attn_layer = attentions[layer_index][0]  # (num_heads, seq_len, seq_len)
    attn_mean = attn_layer.mean(dim=0)       # (seq_len, seq_len)

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    valid_len = int(attention_mask[0].sum().item())
    tokens = tokens[:valid_len]
    attn_mean = attn_mean[:valid_len, :valid_len]

    # RoBERTa uses 'Ġ' prefix for tokens that start a word
    target_subtokens = [f"Ġ{target_word}", target_word]
    target_indices = [i for i, tok in enumerate(tokens) if tok in target_subtokens]

    if not target_indices:
        print(f"Token '{target_word}' not found in: {tokens}")
        return

    # We take the first occurrence of the target word
    idx = target_indices[0]

    # Attention from this token to all others
    attn_vec = attn_mean[idx].cpu().numpy()

    plt.figure(figsize=(10, 3))
    plt.bar(range(len(tokens)), attn_vec)
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.ylabel("Attention weight")
    plt.title(f"Attention FROM '{tokens[idx]}' (layer {layer_index})")
    plt.tight_layout()
    plt.show()

    # Print token-attention pairs sorted by weight
    pairs = list(zip(tokens, attn_vec))
    pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
    print("Top attended tokens from", tokens[idx])
    for tok, val in pairs_sorted[:10]:
        print(f"{tok:15s} -> {val:.3f}")


In [ ]:
sample_text = "Worst product of the year, can't believe they still make stuff as bad as this"
visualize_token_attention(sample_text, model, tokenizer, device,
                          target_word="product", layer_index=-1, max_len=24)

### 4.8 Grad-CAM Visualization

In [ ]:
def grad_cam_importance(text, model, tokenizer, device, max_len=64, target_label=None):
    """
    Compute a Grad-CAM-like importance score for each token in the input text,
    using gradients w.r.t. the input embeddings rather than the final hidden state.
    Returns (tokens, importance_scores).
    """

    model.eval()

    # Tokenize
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_len
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # Get embeddings for each token
    # (batch, seq_len, hidden_dim)
    embeddings = model.roberta.embeddings(input_ids=input_ids)

    # We want gradients with regard to these embeddings
    embeddings = embeddings.detach()
    embeddings.requires_grad_(True)

    # Forward pass using embeddings directly
    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask,
        return_dict=True
    )

    logits = outputs.logits  # (1, num_labels)

    # Choose target label (predicted by default)
    if target_label is None:
        target_label = logits.argmax(dim=-1).item()

    target_logit = logits[0, target_label]

    # Backprop to embeddings
    model.zero_grad()
    target_logit.backward()

    grads = embeddings.grad[0]        # (seq_len, hidden_dim)
    # Using L2 norm of gradient as importance score
    token_importance = grads.norm(dim=-1).detach().cpu().numpy()  # (seq_len,)

    # Mask padding tokens
    mask = attention_mask[0].cpu().numpy().astype(bool)
    token_importance = token_importance[mask]
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0][mask])

    # Normalize to [0, 1]
    token_importance = token_importance - token_importance.min()
    if token_importance.max() > 0:
        token_importance = token_importance / token_importance.max()

    return tokens, token_importance


In [ ]:
def plot_token_importance(tokens, importance, title="Grad-CAM Token Importance"):
    plt.figure(figsize=(10, 2))
    plt.bar(range(len(tokens)), importance)
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.ylabel("Importance")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# GOOD REVIEW #1
sample_text = df_test_roberta["joined_text"].iloc[5]
print("TEXT:\n", sample_text)

tokens, scores = grad_cam_importance(sample_text, model, tokenizer, device, max_len=64)
plot_token_importance(tokens, scores, title="Grad-CAM Importance for Sample Review")


In [ ]:
# GOOD REVIEW #2
sample_text = df_test_roberta["joined_text"].iloc[7]
print("TEXT:\n", sample_text)

tokens, scores = grad_cam_importance(sample_text, model, tokenizer, device, max_len=64)
plot_token_importance(tokens, scores, title="Grad-CAM Importance for Sample Review")

In [ ]:
# GOOD REVIEW #3
sample_text = df_test_roberta["joined_text"].iloc[9]
print("TEXT:\n", sample_text)

tokens, scores = grad_cam_importance(sample_text, model, tokenizer, device, max_len=64)
plot_token_importance(tokens, scores, title="Grad-CAM Importance for Sample Review")

In [ ]:
# BAD REVIEW #1
sample_text = df_test_roberta["joined_text"].iloc[11]
print("TEXT:\n", sample_text)

tokens, scores = grad_cam_importance(sample_text, model, tokenizer, device, max_len=64)
plot_token_importance(tokens, scores, title="Grad-CAM Importance for Sample Review")

In [ ]:
# BAD REVIEW #2
sample_text = df_test_roberta["joined_text"].iloc[13]
print("TEXT:\n", sample_text)

tokens, scores = grad_cam_importance(sample_text, model, tokenizer, device, max_len=64)
plot_token_importance(tokens, scores, title="Grad-CAM Importance for Sample Review")

In [ ]:
# BAD REVIEW #3
sample_text = df_test_roberta["joined_text"].iloc[16]
print("TEXT:\n", sample_text)

tokens, scores = grad_cam_importance(sample_text, model, tokenizer, device, max_len=64)
plot_token_importance(tokens, scores, title="Grad-CAM Importance for Sample Review")

In [ ]:
# Printing the attention weights
for tok, sc in zip(tokens, scores):
    print(f"{tok:15s} -> {sc:.3f}")
